In [ ]:
#!/usr/bin/env python3
"""
adap_algo_bayesien_pbs.py

Pipeline bayésien pour la détection de ciel clair à partir de Landsat.

Sources :
- Observations Landsat (TB10, TB11)
- Simulations RTTOV (TB10_sim, TB11_sim)
- A priori nuage Cloud SAF (P_c P_notc)
- Incertitude SST issue d’ODYSSEA

Objectif :
    Calculer la probabilité a posteriori :
        P(clear | y_obs, x_b)

Sorties :
- CSV (pixel par pixel)
- Carte GeoTIFF P(clear)
"""

# ==========================================================
# 0) IMPORTS
# ==========================================================

import os
import glob
import math
import time
import warnings
from datetime import datetime

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from scipy.ndimage import generic_filter
from sklearn.neighbors import KDTree, BallTree
import rioxarray as rxr

warnings.filterwarnings("ignore", category=RuntimeWarning)

start = time.time()


# ==========================================================
# LUT LSD → P(clear) / P(cloud)
# ==========================================================

xs = np.array([
    0.00, 0.02, 0.04, 0.06, 0.08,
    0.10, 0.12, 0.14, 0.16, 0.18,
    0.20, 0.22, 0.24, 0.26, 0.28,
    0.30, 0.32, 0.34, 0.36, 0.38,
    0.40, 0.42, 0.44, 0.46, 0.48,
    0.50, 0.52, 0.54, 0.56, 0.58,
    0.60, 0.62, 0.64, 0.66, 0.68,
    0.70, 0.72, 0.74, 0.76, 0.78,
    0.80, 0.82, 0.84, 0.86, 0.88,
    0.90, 0.92, 0.94, 0.96, 1.00
])

pdf_clear = np.array([
    0.00, 2.0, 10.0, 6.0, 2.2,
    1.1, 0.85, 0.60, 0.52, 0.48,
    0.45, 0.43, 0.40, 0.39, 0.37,
    0.36, 0.34, 0.33, 0.32, 0.30,
    0.29, 0.28, 0.27, 0.26, 0.25,
    0.24, 0.23, 0.22, 0.21, 0.20,
    0.19, 0.18, 0.17, 0.16, 0.15,
    0.14, 0.14, 0.13, 0.13, 0.12,
    0.12, 0.11, 0.11, 0.10, 0.10,
    0.10, 0.10, 0.09, 0.09, 0.09
])

pdf_cloudy = np.array([
    0.00, 1.0, 3.5, 2.5, 1.8,
    1.4, 1.2, 1.1, 1.05, 1.00,
    0.95, 0.92, 0.90, 0.88, 0.85,
    0.83, 0.80, 0.78, 0.76, 0.74,
    0.72, 0.70, 0.68, 0.67, 0.65,
    0.64, 0.62, 0.61, 0.60, 0.58,
    0.57, 0.56, 0.55, 0.54, 0.53,
    0.52, 0.51, 0.50, 0.49, 0.48,
    0.47, 0.46, 0.45, 0.44, 0.43,
    0.42, 0.41, 0.40, 0.39, 0.38
])

den = pdf_clear + pdf_cloudy
P_cloud_LUT = pdf_cloudy / den

# ==========================================================
# 1) PARAMÈTRES UTILISATEUR
# ==========================================================

# ---------- RTTOV ----------
rttov_data_name = "rttov_LC09_L1C_GRIDDED_DATASET_202027_20221216.nc"
rttov_data_path = "/home/kcorrea/Trishna/data/rttov/" + rttov_data_name

# ---------- Observations Landsat ----------
obs_root = "/home/kcorrea/Trishna/data/l1c/"
obs_file_pattern = rttov_data_name[6:-3] + "*_T1.nc"

# ---------- Cloud SAF ----------
cloud_saf_folder = "/home/kcorrea/Trishna/data/cloud_saf/"

# ---------- ODYSSEA ----------
fichier_ODS = "/home/kcorrea/Trishna/data/ODYSSEA/20221020000000-IFR-L4_GHRSST-SSTfnd-ODYSSEA-ATL_002-v02.1-fv01.0.nc"

# ---------- Sorties ----------
OUT_BASE = "/home/kcorrea/Trishna/bayesian"
os.makedirs(OUT_BASE, exist_ok=True)

# ==========================================================
# SEUILS DE REJET GROS NUAGES
# ==========================================================

TB_MIN_CLEAR = 270.0     # K : nuages épais très froids
D2_MAX = 25.0            # ≈ 5σ (Mahalanobis)


# ==========================================================
# 2) PARAMÈTRES PHYSIQUES
# ==========================================================

NEdT_band10 = 0.06
NEdT_band11 = 0.08
ep10_FM = 0.25
ep11_FM = 0.30
rmseSST = 1.1

R_matrix = np.diag([
    NEdT_band10**2 + ep10_FM**2,
    NEdT_band11**2 + ep11_FM**2
])

# ==========================================================
# 3) PARAMÈTRES TEXTURE
# ==========================================================

window = 15
x0_t = 0.9
k_t = 40
EPS = 1e-12

# ==========================================================
# 4) FONCTIONS UTILITAIRES
# ==========================================================

def ensure_2d(arr):
    """Force un tableau (1,y,x) → (y,x)"""
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] == 1:
        return arr[0]
    return arr


def compute_std_map_nan_safe(arr, window):
    """Écart-type local (texture), ignore les NaN"""
    def nanstd(x):
        return np.nanstd(x)
    return generic_filter(arr, nanstd, size=(window, window), mode="nearest")


def textural_weight(std, x0, k):
    """Poids textural sigmoïde"""
    return 1.0 / (1.0 + np.exp(k * (std - x0)))


def clear_obs_spect_proba(y_obs, yb, H, B, R):
    """Vraisemblance gaussienne multivariée"""
    cov = H.T @ B @ H + R + np.eye(2) * 1e-8
    inv_cov = np.linalg.inv(cov)
    det_cov = max(np.linalg.det(cov), EPS)
    diff = (y_obs - yb).reshape(-1, 1)
    d2 = float(diff.T @ inv_cov @ diff)
    denom = np.sqrt((2*np.pi)**2 * det_cov)
    return float((1/denom) * np.exp(-0.5 * d2))


def posterior_probability(P_c, P_nc, P_y_c, P_y_nc):
    """P(clear | y)"""
    ratio = (P_nc * P_y_nc) / np.maximum(P_c * P_y_c, EPS)
    return 1.0 / (1.0 + ratio)

# ==========================================================
# 5) LECTURE RTTOV
# ==========================================================

ds2 = xr.open_dataset(rttov_data_path)

RTTOV_TB10 = ds2.BtRefl[:, 0].values
RTTOV_TB11 = ds2.BtRefl[:, 1].values
STK10 = ds2.SkinK[:, 0].values
STK11 = ds2.SkinK[:, 1].values
TWCV = ds2.twvc.values
TWCVK10 = ds2.twvcK[:, 0].values
TWCVK11 = ds2.twvcK[:, 1].values
RTTOV_lat = ds2.lat.values
RTTOV_lon = ds2.lon.values

print("RTTOV chargé")

# ==========================================================
# 6) LECTURE OBSERVATIONS LANDSAT
# ==========================================================

obs_file = glob.glob(os.path.join(obs_root, obs_file_pattern))[0]
ds_obs = xr.open_dataset(obs_file)

TB10 = ensure_2d(ds_obs.brightness_temperature_11.values)
TB11 = ensure_2d(ds_obs.brightness_temperature_12.values)
lat = ensure_2d(ds_obs.latitude.values)
lon = ensure_2d(ds_obs.longitude.values)

# ==========================================================

# # masque terre éventuel
# land_mask_values = ds_obs.get("land_mask", None)
# # test land_mask var
# if land_mask_values is not None:
#     print('Masque terre existe pour ce fichier')
#     land_mask = ensure_2d(land_mask_values, 'land_mask')

# if land_mask is not None:
#     land_mask = np.array(land_mask)
#     TB10 = np.where(land_mask == 1, np.nan, TB10)
#     TB11 = np.where(land_mask == 1, np.nan, TB11)

# ==========================================================
# 6bis) MASQUE TERRE / MER
# ==========================================================

land_mask = None

if "land_mask" in ds_obs.variables:
    print("Masque terre existe pour ce fichier")
    land_mask = ensure_2d(ds_obs["land_mask"].values)

if land_mask is not None:
    # Convention : 1 = terre, 0 = eau
    land_mask = np.asarray(land_mask)

    TB10 = np.where(land_mask == 1, np.nan, TB10)
    TB11 = np.where(land_mask == 1, np.nan, TB11)

# ==========================================================
# 7) PIXELS VALIDES
# ==========================================================

mask = np.isfinite(TB10) & np.isfinite(TB11)

TB10_v = TB10[mask]
TB11_v = TB11[mask]
lat_v = lat[mask]
lon_v = lon[mask]

print("Pixels valides :", len(TB10_v))

# ==========================================================
# 8) MATCHING RTTOV ↔ LANDSAT
# ==========================================================

rttov_xy = np.column_stack([RTTOV_lon*np.cos(np.deg2rad(RTTOV_lat)), RTTOV_lat])
landsat_xy = np.column_stack([lon_v*np.cos(np.deg2rad(lat_v)), lat_v])

tree = KDTree(rttov_xy)
_, idx = tree.query(landsat_xy, k=1)
idx = idx.flatten()

# ==========================================================
# 9) CLOUD SAF – A PRIORI
# ==========================================================

mois = obs_file[-22:-20]
file_pre = f"CFCmm2024{mois}*GL.nc"
cloud_files = glob.glob(os.path.join(cloud_saf_folder, file_pre))

if cloud_files:
    ds_cloud = xr.open_dataset(cloud_files[0])
    region = ds_cloud.sel(
        lat=slice(lat.min(), lat.max()),
        lon=slice(lon.min(), lon.max())
    )
    P_not_c = float(region.cma_prob.mean() / 100.0)
else:
    P_not_c = 0.2

P_c = 1.0 - P_not_c
print(f"A priori Cloud SAF → P(clear)={P_c:.2f}")

# ==========================================================
# 10) ODYSSEA – INCERTITUDE SST
# ==========================================================

ds_od = xr.open_dataset(fichier_ODS)
region_od = ds_od.sel(
    lat=slice(lat.min(), lat.max()),
    lon=slice(lon.min(), lon.max())
)

error_od = region_od.analysis_error.squeeze().values
lat_o = region_od.lat.values
lon_o = region_od.lon.values

LAT, LON = np.meshgrid(lat_o, lon_o, indexing="ij")
valid = ~np.isnan(error_od)

points_od = np.vstack([LAT[valid], LON[valid]]).T
tree_od = BallTree(np.deg2rad(points_od), metric="haversine")

points_ls = np.vstack([lat_v, lon_v]).T
_, idx_od = tree_od.query(np.deg2rad(points_ls), k=1)
error_sst = error_od[valid][idx_od.flatten()]

# ==========================================================
# 11) TEXTURE
# ==========================================================

std_map = compute_std_map_nan_safe(TB10, window)
std_v = std_map[mask]
# tex_w = textural_weight(std_v, x0_t, k_t)
# LSD normalisée [0,1]
LSD = np.clip(std_v / x0_t, 0.0, 1.0)
# LUT texture → P(cloud)
P_cloud_tex = np.interp(LSD, xs, P_cloud_LUT)
# Probabilité de ciel clair = 1 - P(cloud)
P_clear_tex = 1.0 - P_cloud_tex

# ==========================================================
# 11 bis) PROTECTION TEXTURE CONTRE GROS NUAGES
# ==========================================================

# D2_TEX_MAX = 9.0   # ≈ 3σ

# tex_w = np.where(d2_all > D2_TEX_MAX, 0.0, tex_w)

# ==========================================================
# 12) PROBABILITÉ SPECTRALE
# ==========================================================

# spect = np.zeros(len(TB10_v))

# for i in range(len(TB10_v)):
#     y_obs = np.array([TB10_v[i], TB11_v[i]])
#     yb = np.array([RTTOV_TB10[idx[i]], RTTOV_TB11[idx[i]]])

#     H = np.array([
#         [STK10[idx[i]], TWCVK10[idx[i]]],
#         [STK11[idx[i]], TWCVK11[idx[i]]]
#     ])

#     deltaTWCV = 0.06029 * TWCV[idx[i]] + 1.167
#     B = np.diag([error_sst[i]**2 + rmseSST**2, deltaTWCV**2])

#     spect[i] = clear_obs_spect_proba(y_obs, yb, H, B, R_matrix)

spect = np.zeros(len(TB10_v))
d2_all = np.zeros(len(TB10_v))  # diagnostic utile

for i in range(len(TB10_v)):
    y_obs = np.array([TB10_v[i], TB11_v[i]])
    yb = np.array([RTTOV_TB10[idx[i]], RTTOV_TB11[idx[i]]])

    H = np.array([
        [STK10[idx[i]], TWCVK10[idx[i]]],
        [STK11[idx[i]], TWCVK11[idx[i]]]
    ])

    deltaTWCV = 0.06029 * TWCV[idx[i]] + 1.167
    B = np.diag([error_sst[i]**2 + rmseSST**2, deltaTWCV**2])

    # # ---------- calcul distance de Mahalanobis ----------
    # cov = H.T @ B @ H + R_matrix + np.eye(2) * 1e-8
    # inv_cov = np.linalg.inv(cov)

    # diff = (y_obs - yb).reshape(-1, 1)
    # d2 = float(diff.T @ inv_cov @ diff)
    # d2_all[i] = d2

    # ---------- REJET DUR GROS NUAGES ----------
    if (TB10_v[i] < TB_MIN_CLEAR) or (TB11_v[i] < TB_MIN_CLEAR): #or (d2 > D2_MAX):
        spect[i] = 0.0
        continue

    # ---------- vraisemblance claire ----------
    spect[i] = clear_obs_spect_proba(y_obs, yb, H, B, R_matrix)

# ==========================================================
# 12 bis) REJET TEXTURE BASE SUR MAHALANOBIS
# ==========================================================

# D2_TEX_MAX = 9.0   # ≈ 3σ
# tex_w = np.where(d2_all > D2_TEX_MAX, 0.0, tex_w)

# ==========================================================
# 13) FUSION & POSTÉRIEUR
# ==========================================================

# floor = np.clip(1 - std_v / x0_t, 0, 1) * 0.6
# P_y_c = np.maximum(spect * tex_w, floor)
# P_y_nc = np.where(std_v <= x0_t & , 1e-5, 1 - 1e-5)

# Pc = posterior_probability(P_c, P_not_c, P_y_c, P_y_nc)


# ==========================================================
# 13) FUSION & POSTÉRIEUR
# ==========================================================

# Plancher clair
# floor = np.clip(1 - std_v / x0_t, 0, 1) * 0.6
floor = EPS

# Vraisemblance clair
# P_y_c = np.maximum(spect * tex_w, floor)
P_y_c = spect * P_clear_tex # ajouté aujourd'hui

# LSD normalisée [0,1]
LSD = np.clip(std_v / x0_t, 0.0, 1.0)

# LUT texture → P(cloud)
#P_cloud_tex = np.interp(LSD, xs, P_cloud_LUT) # commanté aujour'hui

# Vraisemblance nuage
P_y_nc = np.where(
    spect == 0.0,
    1.0,                        # rejet spectral dur
    np.where(
        std_v <= x0_t,
        1e-5,                   # clair homogène
        P_cloud_tex             # LUT LSD
    )
)

# PROBABILITÉ POSTÉRIEURE
Pc = posterior_probability(P_c, P_not_c, P_y_c, P_y_nc)


# ==========================================================
# 14) SORTIES CSV & GEOTIFF
# ==========================================================

out_dir = os.path.join(OUT_BASE, rttov_data_name[-11:-3])
os.makedirs(out_dir, exist_ok=True)

# ---- CSV ----
csv = np.column_stack([lat_v, lon_v, TB10_v, TB11_v, std_v, P_y_c, Pc])
np.savetxt(
    os.path.join(out_dir, "bayesian_clear_probability_landmask_W15_corGrosNuage2pasfloor.csv"),
    csv,
    delimiter=",",
    header="lat,lon,TB10,TB11,std,likelihood,Pc",
    comments=""
)

# ---- GeoTIFF ----
# Pc_map = np.full(TB10.shape, np.nan)
# Pc_map[mask] = Pc

# Pc_xr = xr.DataArray(
#     Pc_map,
#     dims=("y", "x"),
#     coords={"y": lat[:, 0], "x": lon[0, :]},
#     name="P_clear"
# ).rio.write_crs("EPSG:4326")

# Pc_xr.rio.to_raster(os.path.join(out_dir, "Pclear_W7_landmask.tif"))
# ==========================================================
# 14) SORTIE GeoTIFF SUR GRILLE LANDSAT
# ==========================================================

# Reconstruction carte 2D sur la grille Landsat originale
Pc_map = np.full(TB10.shape, np.nan)
Pc_map[mask] = Pc

print("Pc_map shape :", Pc_map.shape)

# ==========================================================
# Cas 1 : lat/lon 2D (grille Landsat réelle)
# ==========================================================

try:
    Pc_xr = xr.DataArray(
        Pc_map,
        dims=("y", "x"),
        coords={
            "y": (("y", "x"), lat),
            "x": (("y", "x"), lon),
        },
        name="P_clear"
    )

    Pc_xr = Pc_xr.rio.write_crs("EPSG:4326", inplace=True)

    tif_out = os.path.join(
        out_dir,
        f"{os.path.basename(obs_file)[:-7]}"
        f"_clear_probability_x0{x0_t}"
        f"_k{k_t}"
        f"_deltaTWCV_variable"
        f"_w{window}_v6f06_corGrosNuage2pasfloor.tif"
    )

    Pc_xr.rio.to_raster(tif_out)
    print(" GeoTIFF sauvegardé (grille Landsat 2D) :", tif_out)

# ==========================================================
# Cas 2 : fallback (coords 1D si nécessaire)
# ==========================================================

except Exception as e:
    print(" Écriture GeoTIFF 2D échouée, fallback activé")
    print("Erreur :", e)

    try:
        lat_1d = lat[:, 0]
        lon_1d = lon[0, :]

        Pc_xr2 = xr.DataArray(
            Pc_map,
            dims=("y", "x"),
            coords={"y": lat_1d, "x": lon_1d},
            name="P_clear"
        )

        Pc_xr2 = Pc_xr2.rio.write_crs("EPSG:4326", inplace=True)

        tif_out2 = os.path.join(
            out_dir,
            f"{os.path.basename(obs_file)[:-7]}"
            f"_clear_probability_x0{x0_t}"
            f"_k{k_t}"
            f"_deltaTWCV_variable"
            f"_w{window}_fallback_v6_corGrosNuage2pasfloor.tif"
        )

        Pc_xr2.rio.to_raster(tif_out2)
        print(" GeoTIFF sauvegardé (fallback 1D) :", tif_out2)

    except Exception as e2:
        print("Échec total écriture GeoTIFF")
        print("Erreur :", e2)

end = time.time()
print(f"Temps d’exécution : {end - start:.2f} s")

print(" CSV et GeoTIFF sauvegardés")
#print(f"Temps total : {time.time() - start:.1f} s")
